# Muestra de 50 preguntas en Colab (GPU) + evaluador oficial

Genera las respuestas a `data/sample_50.jsonl` con el sistema completo (bge-m3 + reranker +
Qwen3-8B, temperatura 0) en una GPU gratuita de Colab y las califica con `scripts/evaluate.py`,
primero sin juez y luego con RAGAS.

**Preparación (una sola vez):**
1. En el computador: `python -m src.index.paquete_colab --modo muestra` → `dist/paquete_colab_muestra.zip`.
2. Subir ese zip a **Google Drive**, en *Mi unidad* (raíz). Si lo pone en otra carpeta, ajuste
   `RUTA_ZIP` en la celda 2.
3. **Llave del juez (para RAGAS):** en Colab, ícono de **llave 🔑 (Secretos)** en la barra
   izquierda → *Agregar nuevo secreto* → nombre `OPENROUTER_API_KEY`, valor = la llave →
   activar *Acceso al cuaderno*. Así la llave no queda escrita en el cuaderno.
4. **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4** → Guardar.

Luego ejecutar las celdas en orden (Shift + Enter).

In [ ]:
# 1. Comprobar GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Traer el paquete desde Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUTA_ZIP = "/content/drive/MyDrive/paquete_colab_muestra.zip"   # ajuste si lo guardó en otra carpeta
!rm -rf /content/proyecto && mkdir -p /content/proyecto
!unzip -q "{RUTA_ZIP}" -d /content/proyecto
%cd /content/proyecto
!ls build/indice data

## 3. Instalar dependencias (≈ 3–5 min)

Se conserva el `torch` con CUDA de Colab. `llama-cpp-python` se instala con soporte CUDA para
correr Qwen3-8B (GGUF de 4 bits) en la GPU. Si no hay rueda precompilada para esta versión de
Python, la celda 3b la compila (≈ 15 min).

In [ ]:
!pip install -q sentence-transformers==6.1.0 transformers==5.17.0 faiss-cpu==1.15.1 bm25s==0.3.11 PyStemmer==3.1.0 python-dotenv==1.2.3 jsonschema==4.26.0 httpx huggingface_hub
!pip install -q llama-cpp-python==0.3.35 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 --only-binary llama-cpp-python || echo "SIN RUEDA PRECOMPILADA: ejecute la celda 3b"

In [ ]:
# 3b. SOLO si la celda anterior dijo "SIN RUEDA PRECOMPILADA" (compila con CUDA, ≈ 15 min)
!CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python==0.3.35 --no-binary llama-cpp-python --force-reinstall --no-deps

In [ ]:
# 3c. Verificar que llama.cpp ve la GPU (debe decir True)
import llama_cpp
print("GPU disponible para llama.cpp:", llama_cpp.llama_supports_gpu_offload())

## 4. Configurar todo en GPU y probar el decoder

In [ ]:
import os
os.environ.update({"ENCODER_DEVICE": "cuda", "DECODER_DEVICE": "cuda", "DECODER_BACKEND": "llamacpp",
                   "CORPUS_SOURCE": "local", "PYTHONIOENCODING": "utf-8",
                   # JAX/TF de Colab reservan el 75 % de la VRAM si se importan: a CPU.
                   "JAX_PLATFORMS": "cpu", "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
                   "TF_FORCE_GPU_ALLOW_GROWTH": "true", "USE_TF": "0", "USE_JAX": "0"})
!python -m src.generation.ping --sin-cache
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 5. Responder las 50 preguntas de la muestra

Escribe cada respuesta al terminarla en `runs/muestra_colab/`. Si Colab se desconecta,
**vuelva a ejecutar esta celda**: continúa donde quedó.

In [ ]:
!python -m src.pipeline.main --split sample --tag muestra_colab; echo "CÓDIGO DE SALIDA: $? (0 = bien; cualquier otro número = error, revise arriba)"
!cat runs/muestra_colab/tiempos.json

## 6. Evaluador oficial sin juez (50 puntos deterministas)

In [ ]:
!python scripts/evaluate.py --submission runs/muestra_colab/submissions.jsonl --split sample --out runs/muestra_colab/reporte.json

## 7. Cargar la llave del juez

Toma `OPENROUTER_API_KEY` de los Secretos de Colab (🔑). Si no está, la pide sin mostrarla.
Nunca se imprime ni se guarda en archivos.

In [ ]:
import os, getpass
try:
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except Exception:
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OPENROUTER_API_KEY: ")
print("Llave cargada:", bool(os.environ.get("OPENROUTER_API_KEY")), "(no se muestra)")

## 8. Evaluador oficial con el juez RAGAS (80 puntos automáticos)

In [ ]:
!pip install -q -r scripts/requirements-evaluador.txt
!python scripts/evaluate.py --submission runs/muestra_colab/submissions.jsonl --split sample --ragas --out runs/muestra_colab/reporte_ragas.json

## 9. Guardar resultados

Copia `runs/muestra_colab/` (respuestas, trazas, tiempos y reportes) a Google Drive y lo
descarga. En el computador, descomprimir `muestra_colab.zip` en la raíz del proyecto.

In [ ]:
!cd /content/proyecto && zip -q -r /content/muestra_colab.zip runs/muestra_colab
!cp /content/muestra_colab.zip /content/drive/MyDrive/
from google.colab import files
files.download("/content/muestra_colab.zip")